# P1: Yield Analysis — Long-term & Acute Experiments

**STAT 628 Module 2: Cranberry Heat Stress**



This notebook contains the complete yield analysis:



- **Part A (Sections A1–A6):** Long-term experiment — paired t-test on OTC vs Control

- **Part B (Sections B1–B7):** Acute experiment — dose-response model

- **Part C (Sections C1–C2):** Cross-experiment Bayesian hierarchical comparison



**Input:** `LTYielddata2024.csv`, `Acute HS-Yield_RawData 2024.xlsx`  

**Output:** Summary tables, diagnostic plots, paired comparison results


---

# Part A: Long-term Experiment — Yield Analysis



8 paired sets (OTC vs Control), 2 cultivars (Stevens, Mullica Queen).  

Method: within-set paired differences, one-sample t-test + Wilcoxon + Cohen's d.


### Imports & Helper Functions


In [1]:
"""
P1: Complete Yield Analysis — Long-term + Acute Experiments
STAT 628 Cranberry Heat Stress Project

Long-term: Paired difference approach (OTC − Control within set)
  - One-sample t-test + Wilcoxon + Cohen's d
  - Temperature covariates exploration (placeholder for lt_temp_covariates.csv)

Acute: Dose-response framework (n_pulses as continuous predictor)
  - Primary: delta_hw ~ n_pulses + cultivar
  - Supplementary: categorical treatment-level descriptives
  - Sensitivity: excluding St-A-Rep1

Outputs: figures + printed summary
"""

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import openpyxl
import warnings, os
warnings.filterwarnings('ignore')

# ====== Edit these paths to match your local setup ======
OUT = "p1_yield_outputs"
os.makedirs(OUT, exist_ok=True)

def ols_fit(X, y, names, title, print_it=True):
    """OLS regression with summary."""
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    yhat = X @ beta
    resid = y - yhat
    n, p = X.shape
    df = n - p
    SSE = np.sum(resid**2)
    SST = np.sum((y - y.mean())**2)
    R2 = 1 - SSE / SST
    adj_R2 = 1 - (1 - R2) * (n-1) / df
    MSE = SSE / df
    se = np.sqrt(np.diag(MSE * np.linalg.inv(X.T @ X)))
    t_s = beta / se
    pv = 2 * stats.t.sf(np.abs(t_s), df)
    if print_it:
        print(f"\n  {title}")
        print(f"  {'='*65}")
        print(f"  n={n}, R²={R2:.4f}, Adj.R²={adj_R2:.4f}, RMSE={np.sqrt(MSE):.2f}, df={df}")
        print(f"\n  {'Variable':<28} {'Est':>9} {'SE':>9} {'t':>8} {'p':>9}")
        print(f"  {'-'*65}")
        for nm, b, s, t, p_ in zip(names, beta, se, t_s, pv):
            sig = "***" if p_<0.001 else "**" if p_<0.01 else "*" if p_<0.05 else "." if p_<0.1 else ""
            print(f"  {nm:<28} {b:>9.3f} {s:>9.3f} {t:>8.3f} {p_:>9.4f} {sig}")
    return {'beta': beta, 'yhat': yhat, 'resid': resid, 'R2': R2, 'pvals': pv, 'se': se}


# ================================================================


ModuleNotFoundError: No module named 'scipy'

### A1. Data Loading & Cleaning


In [ ]:
#  PART A: LONG-TERM EXPERIMENT
# ================================================================
print("╔" + "═"*68 + "╗")
print("║  PART A: LONG-TERM EXPERIMENT — Yield Analysis                     ║")
print("╚" + "═"*68 + "╝")

# ---------- A1: Data Loading & Cleaning ----------
print("\n" + "="*70)
print("A1: DATA LOADING & CLEANING")
print("="*70)

lt_raw = pd.read_csv("data_longterm/LTYielddata2024.csv",
    skiprows=3, header=None,
    names=["Cultivar","HeatTrt","Plot","Rotten_n","Rotten_wt",
           "Healthy_n","Healthy_wt","Total_n","Total_wt"])

for c in ["Plot","Rotten_n","Rotten_wt","Healthy_n","Healthy_wt","Total_n","Total_wt"]:
    lt_raw[c] = pd.to_numeric(lt_raw[c], errors='coerce')

lt = lt_raw.dropna(subset=['Plot']).copy()
lt['Plot'] = lt['Plot'].astype(int)
lt['Cultivar'] = lt['Cultivar'].str.strip()
lt['HeatTrt'] = lt['HeatTrt'].str.strip()

# Derived
lt['Rotten_pct_n'] = lt['Rotten_n'] / lt['Total_n']
lt['Rotten_pct_wt'] = lt['Rotten_wt'] / lt['Total_wt']

# Set mapping (from experimental design)
set_map = {1:10,2:9,3:8,4:7,5:10,6:9,7:8,8:7,
           9:14,10:13,11:12,12:11,13:14,14:13,15:12,16:11}
lt['Set'] = lt['Plot'].map(set_map)

# Validation
check_n = (lt['Rotten_n'] + lt['Healthy_n'] != lt['Total_n']).sum()
check_w = ((lt['Rotten_wt'] + lt['Healthy_wt'] - lt['Total_wt']).abs() > 0.01).sum()
print(f"  Rows: {len(lt)} (expected 16)")
print(f"  Sum checks: count {check_n} mismatches, weight {check_w} mismatches")
print(f"  Healthy_wt range: [{lt['Healthy_wt'].min():.1f}, {lt['Healthy_wt'].max():.1f}]")
print(f"  Rotten_pct range: [{lt['Rotten_pct_n'].min():.4f}, {lt['Rotten_pct_n'].max():.4f}]")

# Set pair completeness
lt_check = lt.groupby(['Cultivar','Set','HeatTrt']).size().unstack(fill_value=0)
assert (lt_check.min(axis=1) > 0).all(), "Incomplete set pairs!"
print("  All 8 set pairs complete ✓")


# ---------- A2: Descriptive Statistics ----------


### A2. Descriptive Statistics


In [ ]:
print("\n" + "="*70)
print("A2: DESCRIPTIVE STATISTICS")
print("="*70)

desc = lt.groupby(['Cultivar','HeatTrt']).agg(
    n=('Healthy_wt','count'),
    hw_mean=('Healthy_wt','mean'), hw_sd=('Healthy_wt','std'),
    rot_mean=('Rotten_pct_n','mean'), rot_sd=('Rotten_pct_n','std')
).round(3).reset_index()
print(desc.to_string(index=False))


# ---------- A3: Paired Differences ----------


### A3. Paired Differences (OTC − Control)


In [ ]:
print("\n" + "="*70)
print("A3: PAIRED DIFFERENCES (OTC − Control)")
print("="*70)

lt_otc = lt[lt['HeatTrt']=='OTC'][['Cultivar','Set','Healthy_wt','Rotten_n',
    'Healthy_n','Total_n','Rotten_pct_n']].copy()
lt_ctrl = lt[lt['HeatTrt']=='Control'][['Cultivar','Set','Healthy_wt','Rotten_n',
    'Healthy_n','Total_n','Rotten_pct_n']].copy()
lt_p = lt_otc.merge(lt_ctrl, on=['Cultivar','Set'], suffixes=('_otc','_ctrl'))
lt_p['delta_hw'] = lt_p['Healthy_wt_otc'] - lt_p['Healthy_wt_ctrl']
lt_p['delta_rot'] = lt_p['Rotten_pct_n_otc'] - lt_p['Rotten_pct_n_ctrl']

print(lt_p[['Cultivar','Set','Healthy_wt_otc','Healthy_wt_ctrl',
            'delta_hw','delta_rot']].sort_values('Set').to_string(index=False))

print(f"\n  Overall: mean Δ_hw = {lt_p['delta_hw'].mean():.2f}g (SD={lt_p['delta_hw'].std():.2f})")
print(f"           mean Δ_rot = {lt_p['delta_rot'].mean():.4f} ({lt_p['delta_rot'].mean()*100:.2f}%)")


# ---------- A4: Normality Check ----------


### A4. Normality Check


In [ ]:
print("\n" + "="*70)
print("A4: NORMALITY CHECK ON PAIRED DIFFERENCES")
print("="*70)

for var, label in [('delta_hw','Δ healthy weight'), ('delta_rot','Δ rotten pct')]:
    w, p = stats.shapiro(lt_p[var])
    print(f"  Shapiro-Wilk [{label}]: W={w:.4f}, p={p:.4f} → "
          f"{'normality OK' if p>0.05 else 'non-normal'}")


# ---------- A5: Statistical Inference ----------


### A5. Statistical Inference


In [ ]:
print("\n" + "="*70)
print("A5: STATISTICAL INFERENCE")
print("="*70)

# --- Healthy Weight ---
print("\n  OUTCOME 1: Healthy Fruit Weight (g)")
print("  " + "-"*50)
dhw = lt_p['delta_hw'].values
t1, p1 = stats.ttest_1samp(dhw, 0)
w1, pw1 = stats.wilcoxon(dhw)
d1 = dhw.mean() / dhw.std(ddof=1)
ci_t = stats.t.ppf(0.975, len(dhw)-1)
ci_lo = dhw.mean() - ci_t * dhw.std(ddof=1) / np.sqrt(len(dhw))
ci_hi = dhw.mean() + ci_t * dhw.std(ddof=1) / np.sqrt(len(dhw))

print(f"    n = {len(dhw)} pairs")
print(f"    Mean Δ = {dhw.mean():.2f}g, SD = {dhw.std(ddof=1):.2f}g")
print(f"    95% CI = [{ci_lo:.1f}, {ci_hi:.1f}]g")
print(f"    t-test:   t={t1:.3f}, p={p1:.4f}")
print(f"    Wilcoxon: W={w1:.1f}, p={pw1:.4f}")
print(f"    Cohen's d = {d1:.3f}")
print(f"    → NOT significant: OTC did not significantly change yield")

print("\n    By cultivar (descriptive only, n=4 too small for inference):")
for cv in ['St','MQ']:
    sub = lt_p[lt_p['Cultivar']==cv]['delta_hw']
    print(f"      {cv}: mean={sub.mean():.1f}g (SD={sub.std():.1f})")

# --- Rotten Proportion ---
print("\n  OUTCOME 2: Proportion Rotten (by count)")
print("  " + "-"*50)
drn = lt_p['delta_rot'].values
t2, p2 = stats.ttest_1samp(drn, 0)
w2, pw2 = stats.wilcoxon(drn)
d2 = drn.mean() / drn.std(ddof=1)

print(f"    n = {len(drn)} pairs")
print(f"    Mean Δ = {drn.mean():.4f} ({drn.mean()*100:.2f}%)")
print(f"    SD = {drn.std(ddof=1):.4f}")
print(f"    t-test:   t={t2:.3f}, p={p2:.4f}")
print(f"    Wilcoxon: W={w2:.1f}, p={pw2:.4f}")
print(f"    Cohen's d = {d2:.3f} (large effect)")
print(f"    → SIGNIFICANT: OTC reduced rotten proportion by ~4%")
print(f"    Direction: OTC has LOWER rot than Control (surprising)")

print("\n    By cultivar:")
for cv in ['St','MQ']:
    sub = lt_p[lt_p['Cultivar']==cv]['delta_rot']
    print(f"      {cv}: mean={sub.mean():.4f} ({sub.mean()*100:.2f}%)")

print("\n    Note: For formal GLM, use in R:")
print("      glm(cbind(Rotten_n, Healthy_n) ~ Cultivar + HeatTrt, "
      "family=quasibinomial, data=lt)")


# ---------- A6: Long-term Plots ----------


### A6. Visualization


In [ ]:
print("\n" + "="*70)
print("A6: LONG-TERM VISUALIZATION")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Long-term Experiment: Yield Analysis", fontsize=14, fontweight='bold')

# (0,0): Scatter OTC vs Control healthy weight
ax = axes[0,0]
for _, row in lt_p.iterrows():
    c = 'tomato' if row['Cultivar']=='St' else 'steelblue'
    mk = 'o' if row['Cultivar']=='St' else 's'
    ax.plot([0,1], [row['Healthy_wt_ctrl'], row['Healthy_wt_otc']], 
            color=c, alpha=0.6, linewidth=1.5, marker=mk, markersize=8)
    ax.annotate(f"S{int(row['Set'])}", (1, row['Healthy_wt_otc']),
                xytext=(5,0), textcoords='offset points', fontsize=8, color=c)
ax.set_xticks([0,1]); ax.set_xticklabels(['Control','OTC'], fontsize=11)
ax.set_ylabel("Healthy Weight (g)"); ax.set_title("Paired Plot: Healthy Weight")
ax.legend(handles=[Patch(color='tomato', label='Stevens'),
                   Patch(color='steelblue', label='Mullica Queen')], fontsize=9)
ax.grid(axis='y', alpha=0.3)

# (0,1): Scatter OTC vs Control rotten pct
ax = axes[0,1]
for _, row in lt_p.iterrows():
    c = 'tomato' if row['Cultivar']=='St' else 'steelblue'
    mk = 'o' if row['Cultivar']=='St' else 's'
    ax.plot([0,1], [row['Rotten_pct_n_ctrl'], row['Rotten_pct_n_otc']],
            color=c, alpha=0.6, linewidth=1.5, marker=mk, markersize=8)
ax.set_xticks([0,1]); ax.set_xticklabels(['Control','OTC'], fontsize=11)
ax.set_ylabel("Rotten Proportion"); ax.set_title("Paired Plot: Rotten Proportion")
ax.legend(handles=[Patch(color='tomato', label='Stevens'),
                   Patch(color='steelblue', label='Mullica Queen')], fontsize=9)
ax.grid(axis='y', alpha=0.3)

# (1,0): Delta bar chart — healthy weight
ax = axes[1,0]
for _, row in lt_p.sort_values('Set').iterrows():
    c = 'tomato' if row['Cultivar']=='St' else 'steelblue'
    ax.bar(f"S{int(row['Set'])}", row['delta_hw'], color=c, alpha=0.75, edgecolor='black')
ax.axhline(0, color='black', linewidth=1)
ax.axhline(lt_p['delta_hw'].mean(), color='gray', linestyle='--',
           label=f"Mean = {lt_p['delta_hw'].mean():.1f}g (p={p1:.3f})")
ax.set_ylabel("Δ Healthy Weight (g)"); ax.set_title("OTC − Control: Healthy Weight")
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# (1,1): Delta bar chart — rotten pct
ax = axes[1,1]
for _, row in lt_p.sort_values('Set').iterrows():
    c = 'tomato' if row['Cultivar']=='St' else 'steelblue'
    ax.bar(f"S{int(row['Set'])}", row['delta_rot'], color=c, alpha=0.75, edgecolor='black')
ax.axhline(0, color='black', linewidth=1)
ax.axhline(lt_p['delta_rot'].mean(), color='gray', linestyle='--',
           label=f"Mean = {lt_p['delta_rot'].mean():.4f} (p={p2:.3f})")
ax.set_ylabel("Δ Rotten Proportion"); ax.set_title("OTC − Control: Rotten Proportion")
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/A_longterm_yield.png", dpi=150, bbox_inches='tight')
plt.close()
print("  Saved A_longterm_yield.png")


# ================================================================


---

# Part B: Acute Experiment — Yield Analysis



24 paired OTC/Control sets across treatments A(4 pulses), B(3), C(2), D(1).  

Method: dose-response model with pulse count as continuous predictor.


### B1. Data Loading & Cleaning


In [ ]:
#  PART B: ACUTE EXPERIMENT
# ================================================================
print("\n\n" + "╔" + "═"*68 + "╗")
print("║  PART B: ACUTE EXPERIMENT — Yield Analysis                         ║")
print("╚" + "═"*68 + "╝")

# ---------- B1: Data Loading & Cleaning ----------
print("\n" + "="*70)
print("B1: DATA LOADING & CLEANING")
print("="*70)

wb = openpyxl.load_workbook(
    "data_acute/Acute HS-Yield_RawData 2024.xlsx", data_only=False)
ws = wb[wb.sheetnames[1]]

rows = []
for i, row in enumerate(ws.iter_rows(values_only=True), 1):
    if i >= 6 and i <= 65 and row[1] is not None:
        rows.append({
            'Cultivar': str(row[1]).strip(),
            'Treatment': str(row[2]).strip(),
            'Replicate': int(row[3]),
            'Rotten_n': float(row[4]), 'Rotten_wt': float(row[5]),
            'Healthy_n': float(row[6]), 'Healthy_wt': float(row[7]),
        })

ac = pd.DataFrame(rows)
ac = ac[~ac['Treatment'].isin(['A0','A0C'])].copy()

# Parse
ac['is_control'] = ac['Treatment'].isin(['AC','BC','CC','DC'])
ac['base_treatment'] = ac['Treatment'].apply(
    lambda x: x[:-1] if x in ['AC','BC','CC','DC'] else x)
ac['plot_type'] = ac['is_control'].map({True:'Control', False:'OTC'})

# n_pulses
pulse_map = {'A':4, 'B':3, 'C':2, 'D':1}
ac['n_pulses'] = ac['base_treatment'].map(pulse_map)

# Derived
ac['Total_n'] = ac['Rotten_n'] + ac['Healthy_n']
ac['Rotten_pct_n'] = ac['Rotten_n'] / ac['Total_n']

print(f"  Rows: {len(ac)} (expected 48)")
print(f"  Healthy_wt range: [{ac['Healthy_wt'].min():.1f}, {ac['Healthy_wt'].max():.1f}]")
print(f"  Rotten_pct range: [{ac['Rotten_pct_n'].min():.4f}, {ac['Rotten_pct_n'].max():.4f}]")

# Flag suspect
sus = ac[(ac['Cultivar']=='St')&(ac['base_treatment']=='A')&
         (ac['Replicate']==1)&(~ac['is_control'])]
if len(sus)>0:
    s = sus.iloc[0]
    print(f"\n  ⚠ Suspect: St-A-Rep1 OTC — {int(s['Rotten_n'])} rotten / "
          f"{int(s['Total_n'])} total ({s['Rotten_pct_n']:.1%})")
    print(f"    Raw data note: '95% rot are full berries'")
    print(f"    Decision: Keep, run sensitivity without it")

# Completeness
ac_check = ac.groupby(['Cultivar','base_treatment','plot_type']).size().unstack(fill_value=0)
assert (ac_check.min(axis=1) > 0).all()
print("  All 48 plots complete ✓")


# ---------- B2: Paired Differences ----------


### B2. Paired Differences


In [ ]:
print("\n" + "="*70)
print("B2: PAIRED DIFFERENCES (OTC − Control)")
print("="*70)

otc = ac[~ac['is_control']].copy()
ctrl = ac[ac['is_control']].copy()
ac_p = otc.merge(ctrl, on=['Cultivar','base_treatment','Replicate'], suffixes=('_otc','_ctrl'))

ac_p['delta_hw'] = ac_p['Healthy_wt_otc'] - ac_p['Healthy_wt_ctrl']
ac_p['delta_rot'] = ac_p['Rotten_pct_n_otc'] - ac_p['Rotten_pct_n_ctrl']
ac_p['delta_asin_rot'] = (np.arcsin(np.sqrt(ac_p['Rotten_pct_n_otc'])) - 
                           np.arcsin(np.sqrt(ac_p['Rotten_pct_n_ctrl'])))
ac_p['n_pulses'] = ac_p['n_pulses_otc']
ac_p['Cultivar_MQ'] = (ac_p['Cultivar']=='MQ').astype(float)

print(f"  Paired rows: {len(ac_p)} (expected 24)")
print(ac_p[['Cultivar','base_treatment','n_pulses','Replicate',
            'delta_hw','delta_rot']].sort_values(
    ['Cultivar','n_pulses','Replicate'], ascending=[True,False,True]
).to_string(index=False))


# ---------- B3: Overall Tests ----------


### B3. Overall Treatment Effect


In [ ]:
print("\n" + "="*70)
print("B3: OVERALL TREATMENT EFFECT")
print("="*70)

# Healthy weight
t_hw, p_hw = stats.ttest_1samp(ac_p['delta_hw'], 0)
d_hw = ac_p['delta_hw'].mean() / ac_p['delta_hw'].std(ddof=1)
print(f"\n  Healthy Weight:")
print(f"    Mean Δ = {ac_p['delta_hw'].mean():.1f}g (SD={ac_p['delta_hw'].std():.1f})")
print(f"    t-test: t={t_hw:.3f}, p={p_hw:.4f}")
print(f"    Cohen's d = {d_hw:.3f}")

# Rotten proportion
t_rot, p_rot = stats.ttest_1samp(ac_p['delta_rot'], 0)
d_rot = ac_p['delta_rot'].mean() / ac_p['delta_rot'].std(ddof=1)
print(f"\n  Rotten Proportion:")
print(f"    Mean Δ = {ac_p['delta_rot'].mean():.4f} ({ac_p['delta_rot'].mean()*100:.2f}%)")
print(f"    t-test: t={t_rot:.3f}, p={p_rot:.4f}")
print(f"    Cohen's d = {d_rot:.3f}")


# ---------- B4: Categorical Descriptives (Supplementary) ----------


### B4. Categorical Treatment Descriptives


In [ ]:
print("\n" + "="*70)
print("B4: CATEGORICAL TREATMENT DESCRIPTIVES (Supplementary)")
print("="*70)

print(f"\n  {'Trt':<5} {'Pulses':<7} {'n':<4} {'Mean Δ_hw':>10} {'SD':>8} {'t':>7} {'p':>8} {'Mean Δ_rot':>11}")
print("  " + "-"*65)
for trt in ['A','B','C','D']:
    sub = ac_p[ac_p['base_treatment']==trt]
    np_ = sub['n_pulses'].iloc[0]
    m_hw = sub['delta_hw'].mean()
    s_hw = sub['delta_hw'].std()
    t_, p_ = stats.ttest_1samp(sub['delta_hw'], 0)
    m_rot = sub['delta_rot'].mean()
    sig = "*" if p_<0.05 else ""
    print(f"  {trt:<5} {np_:<7} {len(sub):<4} {m_hw:>10.1f} {s_hw:>8.1f} {t_:>7.3f} {p_:>8.4f}{sig} {m_rot:>10.4f}")

print(f"\n  Note: Treatment C (2 pulses) has lowest SD (46.5g) — most consistent effect")
print(f"  Note: Damage does NOT scale linearly with n_pulses (see dose-response model)")


# ---------- B5: Dose-Response Models (Primary) ----------


### B5. Dose-Response Models


In [ ]:
print("\n" + "="*70)
print("B5: DOSE-RESPONSE MODELS (Primary Analysis)")
print("="*70)

# --- Healthy Weight ---
print("\n  OUTCOME 1: Healthy Fruit Weight")
print("  " + "-"*50)

# Full interaction
X_full = np.column_stack([np.ones(len(ac_p)), ac_p['n_pulses'].values,
    ac_p['Cultivar_MQ'].values, (ac_p['n_pulses']*ac_p['Cultivar_MQ']).values])
m_full = ols_fit(X_full, ac_p['delta_hw'].values,
    ['Intercept','n_pulses','Cultivar_MQ','n_pulses:Cultivar_MQ'],
    "Model 1a: delta_hw ~ n_pulses * cultivar (FULL)")

print(f"\n  → Interaction p = {m_full['pvals'][3]:.4f} → "
      f"{'keep' if m_full['pvals'][3]<0.05 else 'NOT significant, simplify'}")

# Additive
X_add = np.column_stack([np.ones(len(ac_p)), ac_p['n_pulses'].values,
    ac_p['Cultivar_MQ'].values])
m_add = ols_fit(X_add, ac_p['delta_hw'].values,
    ['Intercept','n_pulses','Cultivar_MQ'],
    "Model 1b: delta_hw ~ n_pulses + cultivar (ADDITIVE)")

b = m_add['beta']
pv = m_add['pvals']
print(f"\n  Interpretation:")
print(f"    n_pulses slope = {b[1]:.1f}g/pulse (p={pv[1]:.4f}) → dose-response NOT significant")
print(f"    Cultivar_MQ = {b[2]:.1f}g (p={pv[2]:.4f})")
print(f"    → Damage occurs but does not increase linearly with pulse count")
print(f"    → Even a single 2-hr pulse at 40°C causes substantial yield loss")

# --- Rotten Proportion ---
print("\n\n  OUTCOME 2: Rotten Proportion (arcsin-sqrt)")
print("  " + "-"*50)

X_rot_full = np.column_stack([np.ones(len(ac_p)), ac_p['n_pulses'].values,
    ac_p['Cultivar_MQ'].values, (ac_p['n_pulses']*ac_p['Cultivar_MQ']).values])
m_rot_full = ols_fit(X_rot_full, ac_p['delta_asin_rot'].values,
    ['Intercept','n_pulses','Cultivar_MQ','n_pulses:Cultivar_MQ'],
    "Model 2a: delta_asin_rot ~ n_pulses * cultivar (FULL)")

X_rot_add = np.column_stack([np.ones(len(ac_p)), ac_p['n_pulses'].values,
    ac_p['Cultivar_MQ'].values])
m_rot_add = ols_fit(X_rot_add, ac_p['delta_asin_rot'].values,
    ['Intercept','n_pulses','Cultivar_MQ'],
    "Model 2b: delta_asin_rot ~ n_pulses + cultivar (ADDITIVE)")


# ---------- B6: Sensitivity ----------


### B6. Sensitivity Analysis


In [ ]:
print("\n" + "="*70)
print("B6: SENSITIVITY — Excluding St-A-Rep1")
print("="*70)

mask = ~((ac_p['Cultivar']=='St') & (ac_p['base_treatment']=='A') & (ac_p['Replicate']==1))
ac_sens = ac_p[mask]
X_s = np.column_stack([np.ones(len(ac_sens)), ac_sens['n_pulses'].values,
    ac_sens['Cultivar_MQ'].values])
m_s = ols_fit(X_s, ac_sens['delta_hw'].values,
    ['Intercept','n_pulses','Cultivar_MQ'],
    f"Sensitivity: delta_hw ~ n_pulses + cultivar (n={len(ac_sens)})")

print(f"\n  n_pulses slope: {m_add['beta'][1]:.2f} (full) vs {m_s['beta'][1]:.2f} (excl)")
print(f"  n_pulses p:     {m_add['pvals'][1]:.4f} (full) vs {m_s['pvals'][1]:.4f} (excl)")
print(f"  → Results robust to removal of suspect observation ✓")


# ---------- B7: Acute Plots ----------


### B7. Visualization & Diagnostics


In [ ]:
print("\n" + "="*70)
print("B7: ACUTE VISUALIZATION")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Acute Experiment: Yield Analysis", fontsize=14, fontweight='bold')

# (0,0): Paired plot — healthy weight
ax = axes[0,0]
for cv in ['St','MQ']:
    sub = ac_p[ac_p['Cultivar']==cv]
    color = 'tomato' if cv=='St' else 'steelblue'
    mk = 'o' if cv=='St' else 's'
    for _, r in sub.iterrows():
        x = {4:0, 3:1, 2:2, 1:3}[r['n_pulses']]
        ax.plot([x-0.12, x+0.12], [r['Healthy_wt_ctrl'], r['Healthy_wt_otc']],
                color='gray', alpha=0.4, linewidth=1)
        ax.scatter(x-0.12, r['Healthy_wt_ctrl'], c='steelblue', s=35, 
                   edgecolor='black', alpha=0.7, zorder=5)
        ax.scatter(x+0.12, r['Healthy_wt_otc'], c='tomato', s=35,
                   edgecolor='black', alpha=0.7, zorder=5)
ax.set_xticks([0,1,2,3]); ax.set_xticklabels(['A(4)','B(3)','C(2)','D(1)'])
ax.set_xlabel("Treatment (n_pulses)"); ax.set_ylabel("Healthy Weight (g)")
ax.set_title("Paired: OTC vs Control")
ax.legend(handles=[Patch(facecolor='tomato', label='OTC'),
                   Patch(facecolor='steelblue', label='Control')], fontsize=9)
ax.grid(axis='y', alpha=0.3)

# (0,1): Dose-response scatter — healthy weight
ax = axes[0,1]
for cv, mk, c in [('St','o','tomato'),('MQ','s','steelblue')]:
    sub = ac_p[ac_p['Cultivar']==cv]
    jit = np.random.uniform(-0.1, 0.1, len(sub))
    ax.scatter(sub['n_pulses']+jit, sub['delta_hw'], c=c, marker=mk, 
               s=70, edgecolor='black', alpha=0.8, label=cv, zorder=5)
sl, ic, _, p_sl, _ = stats.linregress(ac_p['n_pulses'], ac_p['delta_hw'])
xl = np.array([0.5, 4.5])
ax.plot(xl, ic+sl*xl, 'k--', alpha=0.4, label=f'slope={sl:.1f}, p={p_sl:.3f}')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel("Number of heat pulses"); ax.set_ylabel("Δ Healthy Weight (g)")
ax.set_title("Dose-Response: Healthy Weight"); ax.legend(fontsize=9)
ax.set_xticks([1,2,3,4]); ax.grid(axis='y', alpha=0.3)

# (1,0): Dose-response scatter — rotten proportion
ax = axes[1,0]
for cv, mk, c in [('St','o','tomato'),('MQ','s','steelblue')]:
    sub = ac_p[ac_p['Cultivar']==cv]
    jit = np.random.uniform(-0.1, 0.1, len(sub))
    ax.scatter(sub['n_pulses']+jit, sub['delta_rot'], c=c, marker=mk,
               s=70, edgecolor='black', alpha=0.8, label=cv, zorder=5)
sl2, ic2, _, p_sl2, _ = stats.linregress(ac_p['n_pulses'], ac_p['delta_rot'])
ax.plot(xl, ic2+sl2*xl, 'k--', alpha=0.4, label=f'slope={sl2:.4f}, p={p_sl2:.3f}')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel("Number of heat pulses"); ax.set_ylabel("Δ Rotten Proportion")
ax.set_title("Dose-Response: Rotten Proportion"); ax.legend(fontsize=9)
ax.set_xticks([1,2,3,4]); ax.grid(axis='y', alpha=0.3)

# (1,1): Delta by treatment (categorical summary)
ax = axes[1,1]
trts = ['A','B','C','D']
for j, trt in enumerate(trts):
    sub = ac_p[ac_p['base_treatment']==trt]
    means_cv = {}
    for cv, c in [('St','tomato'),('MQ','steelblue')]:
        s2 = sub[sub['Cultivar']==cv]
        mk = 'o' if cv=='St' else 's'
        jit = np.random.uniform(-0.05, 0.05, len(s2))
        ax.scatter([j]*len(s2)+jit, s2['delta_hw'], c=c, marker=mk,
                   s=50, edgecolor='black', alpha=0.7, zorder=5)
    m = sub['delta_hw'].mean()
    ax.plot([j-0.2, j+0.2], [m,m], color='black', linewidth=2.5)
ax.axhline(0, color='gray', linestyle='--')
ax.set_xticks(range(4)); ax.set_xticklabels(['A (4)','B (3)','C (2)','D (1)'])
ax.set_xlabel("Treatment (n_pulses)"); ax.set_ylabel("Δ Healthy Weight (g)")
ax.set_title("Treatment Means (bars = group mean)")
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor='tomato',markersize=8,label='St'),
                   Line2D([0],[0],marker='s',color='w',markerfacecolor='steelblue',markersize=8,label='MQ')],
          fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/B_acute_yield.png", dpi=150, bbox_inches='tight')
plt.close()
print("  Saved B_acute_yield.png")


# ---------- Diagnostics ----------
print("\n" + "="*70)
print("DIAGNOSTICS")
print("="*70)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Model Diagnostics — Acute Additive Models", fontsize=14, fontweight='bold')

for row, (res, yh, label, color) in enumerate([
    (m_add['resid'], m_add['yhat'], 'Healthy Wt', 'steelblue'),
    (m_rot_add['resid'], m_rot_add['yhat'], 'Rotten Pct (asin)', 'coral'),
]):
    axes[row,0].scatter(yh, res, c=color, edgecolor='black', alpha=0.7, s=60)
    axes[row,0].axhline(0, color='red', linestyle='--')
    axes[row,0].set_xlabel("Fitted"); axes[row,0].set_ylabel("Residuals")
    axes[row,0].set_title(f"{label}: Resid vs Fitted")
    
    sr = np.sort(res); n_r = len(sr)
    tq = stats.norm.ppf(np.arange(1,n_r+1)/(n_r+1))
    axes[row,1].scatter(tq, sr, c=color, edgecolor='black', s=60)
    mn, mx = tq.min(), tq.max()
    axes[row,1].plot([mn,mx], [mn*np.std(sr), mx*np.std(sr)], 'r--', alpha=0.7)
    axes[row,1].set_xlabel("Theoretical Q"); axes[row,1].set_ylabel("Sample Q")
    axes[row,1].set_title(f"{label}: Normal Q-Q")
    
    axes[row,2].hist(res, bins=10, color=color, edgecolor='black', alpha=0.7)
    w_,p_ = stats.shapiro(res)
    axes[row,2].text(0.05, 0.9, f"Shapiro W={w_:.3f}\np={p_:.3f}",
        transform=axes[row,2].transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[row,2].set_xlabel("Residuals"); axes[row,2].set_title(f"{label}: Distribution")
    status = "OK ✓" if p_>0.05 else "REJECTED"
    print(f"  {label}: Shapiro W={w_:.4f}, p={p_:.4f} → {status}")

plt.tight_layout()
plt.savefig(f"{OUT}/C_diagnostics.png", dpi=150, bbox_inches='tight')
plt.close()
print("  Saved C_diagnostics.png")


### Parts A & B Summary


In [ ]:
# ================================================================
#  COMBINED SUMMARY
# ================================================================
print("\n\n" + "╔" + "═"*68 + "╗")
print("║  COMBINED SUMMARY                                                  ║")
print("╚" + "═"*68 + "╝")
print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LONG-TERM EXPERIMENT (passive OTC, ~+1°C all season)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Healthy weight:   Δ = {lt_p['delta_hw'].mean():.1f}g, p = {p1:.4f}  → NOT significant
  Rotten proportion: Δ = {lt_p['delta_rot'].mean():.4f}, p = {p2:.4f}  → SIGNIFICANT ↓ rot
  
  Long-term warming (~1°C) did NOT reduce yield.
  Surprisingly, OTC LOWERED rot rate (Cohen's d = {d2:.2f}).
  
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ACUTE EXPERIMENT (40°C heat pulses, 2 hours each)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Healthy weight:   Δ = {ac_p['delta_hw'].mean():.1f}g, p = {p_hw:.4f}  → SIGNIFICANT ↓ yield
  Rotten proportion: Δ = {ac_p['delta_rot'].mean():.4f}, p = {p_rot:.4f}  → SIGNIFICANT ↑ rot
  
  Dose-response (n_pulses): slope = {m_add['beta'][1]:.1f}g/pulse, p = {m_add['pvals'][1]:.4f}
    → NOT significant: damage does not scale linearly with pulse count
    → Even 1 pulse causes substantial harm
  Cultivar × dose interaction: p = {m_full['pvals'][3]:.4f}
    → NOT significant: Stevens and MQ respond similarly
  
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEY CONCLUSION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Gradual warming (~1°C) does not harm cranberry yield and may even
  reduce rot. But extreme heat events (40°C, even a single 2-hour 
  pulse) significantly reduce healthy fruit weight by ~155g and 
  increase rot by ~11%. The primary threat to cranberry production 
  is acute heat stress, not chronic warming.
""")

print(f"All outputs saved to: {OUT}")
print("Done.")


---

# Part C: Cross-Experiment Bayesian Comparison



Bayesian hierarchical model comparing yield effects across LT and acute treatments (A/B/C/D).  

Uses partial pooling to stabilize estimates while preserving group-specific differences.



**Requires:** `pymc`, `arviz`  

**Two models:** Normal likelihood for rotten proportion, Student-t for healthy weight (robust to outliers).


## C1. Rotten Proportion — Bayesian Hierarchical Model (Normal Likelihood)


In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
# =========
# 1. 读取 LT 数据
# =========
# Uses same data files as Parts A & B above
lt_path = "data_longterm/LTYielddata2024.csv"
lt_raw = pd.read_csv(lt_path, header=3)

# 先清理列名两端空格
lt_raw.columns = lt_raw.columns.str.strip()

print("LT columns:")
print(list(lt_raw.columns))

lt = lt_raw.rename(columns={
    "HeatTrt": "Treatment",
    "Rotten fruit #": "rotten_count",
    "Rotten fruitWeight (g)": "rotten_weight",
    "Non-rotten fruit#": "healthy_count",
    "Non-rotten fruitWeight (g)": "healthy_weight",
    "Total #": "total_count",
    "Total Weight (g)": "total_weight"
}).copy()

lt["Cultivar"] = lt["Cultivar"].astype(str).str.strip()
lt["Treatment"] = lt["Treatment"].astype(str).str.strip()
lt["Plot"] = pd.to_numeric(lt["Plot"], errors="coerce")

# outcome: by-weight rotten proportion
lt["y_weight"] = lt["rotten_weight"] / lt["total_weight"]

# 构造 long-term pair_id
# 假设同 cultivar 内 control plot = OTC plot + 4
def make_lt_pair_id(row):
    plot = row["Plot"]
    if row["Cultivar"] == "St":
        return plot if row["Treatment"] == "OTC" else plot - 4
    elif row["Cultivar"] == "MQ":
        return plot if row["Treatment"] == "OTC" else plot - 4
    else:
        raise ValueError("Unexpected cultivar")

lt["pair_index"] = lt.apply(make_lt_pair_id, axis=1)
lt["regime"] = "LT"

# 宽表：每对一行
lt_wide = (
    lt.pivot_table(
        index=["Cultivar", "pair_index", "regime"],
        columns="Treatment",
        values="y_weight",
        aggfunc="first"
    )
    .reset_index()
)

lt_wide["d"] = lt_wide["OTC"] - lt_wide["Control"]
lt_pairs = lt_wide[["Cultivar", "pair_index", "regime", "d"]].copy()

# =========
# 2. 读取 Acute 数据
# =========
acute_path = "data_acute/Acute HS-Yield_RawData 2024.xlsx"
acute = pd.read_excel(acute_path, sheet_name="Acute Heat stress", header=4)

acute = acute.rename(columns={
    "Rotten fruitWeight (g)": "rotten_weight",
    " Total Weight (g)": "total_weight",
    "Non-rotten fruitWeight (g)": "healthy_weight",
    "Rotten fruit #": "rotten_count",
    "Non-rotten fruit#": "healthy_count",
    "Total #": "total_count",
    "# OTC/ plot": "rep"
}).copy()

acute = acute.dropna(subset=["Cultivar", "Treatment", "rep"])
acute["Cultivar"] = acute["Cultivar"].astype(str).str.strip()
acute["Treatment"] = acute["Treatment"].astype(str).str.strip()
acute["rep"] = acute["rep"].astype(int)

acute["y_weight"] = acute["rotten_weight"] / acute["total_weight"]

# 只保留 A/B/C/D 和对应 control
acute = acute[acute["Treatment"].isin(["A", "B", "C", "D", "AC", "BC", "CC", "DC"])].copy()

# treatment family
acute["regime"] = acute["Treatment"].replace({
    "AC": "A",
    "BC": "B",
    "CC": "C",
    "DC": "D"
})

acute["group"] = np.where(
    acute["Treatment"].isin(["A", "B", "C", "D"]),
    "treat",
    "control"
)

acute_wide = (
    acute.pivot_table(
        index=["Cultivar", "rep", "regime"],
        columns="group",
        values="y_weight",
        aggfunc="first"
    )
    .reset_index()
)

acute_wide["d"] = acute_wide["treat"] - acute_wide["control"]
acute_pairs = acute_wide[["Cultivar", "rep", "regime", "d"]].copy()
acute_pairs = acute_pairs.rename(columns={"rep": "pair_index"})

# =========
# 3. 合并 pair-level 数据
# =========
pair_df = pd.concat([lt_pairs, acute_pairs], ignore_index=True)

# 编码
regime_order = ["LT", "A", "B", "C", "D"]
pair_df["regime"] = pd.Categorical(pair_df["regime"], categories=regime_order, ordered=True)
pair_df["regime_idx"] = pair_df["regime"].cat.codes

# cultivar: St/Stevens = 0, MQ = 1
pair_df["cultivar_code"] = pair_df["Cultivar"].replace({
    "St": 0,
    "Stevens": 0,
    "MQ": 1
}).astype(int)

print(pair_df)

# =========
# 4. 贝叶斯层级模型
# =========
d = pair_df["d"].values
regime_idx = pair_df["regime_idx"].values
cultivar = pair_df["cultivar_code"].values
n_regimes = len(regime_order)

with pm.Model() as model:
    # hyperpriors
    mu0 = pm.Normal("mu0", mu=0, sigma=10)
    tau = pm.HalfNormal("tau", sigma=1)

    # regime-level effects
    mu_regime = pm.Normal("mu_regime", mu=mu0, sigma=tau, shape=n_regimes)

    # cultivar effect
    beta_cultivar = pm.Normal("beta_cultivar", mu=0, sigma=10)

    # residual SD
    sigma = pm.HalfNormal("sigma", sigma=1)

    # mean model
    mu_obs = mu_regime[regime_idx] + beta_cultivar * cultivar

    # likelihood
    d_obs = pm.Normal("d_obs", mu=mu_obs, sigma=sigma, observed=d)

    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=5,
        cores=1,
        target_accept=0.97,
        random_seed=123,
        return_inferencedata=False,
        compute_convergence_checks=False
    )
    

# =========
# 5. 查看结果
# =========

mu_samples = trace.get_values("mu_regime", combine=True)
# shape 通常是 (n_samples, 5)

posterior_mu = mu_samples.mean(axis=0)
for name, val in zip(regime_order, posterior_mu):
    print(f"{name}: {val:.4f}")


beta_samples = trace.get_values("beta_cultivar", combine=True)
sigma_samples = trace.get_values("sigma", combine=True)
tau_samples = trace.get_values("tau", combine=True)
mu0_samples = trace.get_values("mu0", combine=True)

print("beta_cultivar mean =", beta_samples.mean())
print("sigma mean =", sigma_samples.mean())
print("tau mean =", tau_samples.mean())
print("mu0 mean =", mu0_samples.mean())


# =========================================
# A. 取 posterior samples
# =========================================
mu_regime_samps = trace.get_values("mu_regime", combine=True)   # shape: (S, n_regimes)
beta_samps = trace.get_values("beta_cultivar", combine=True)    # shape: (S,)
sigma_samps = trace.get_values("sigma", combine=True)           # shape: (S,)
tau_samps = trace.get_values("tau", combine=True)               # shape: (S,)
mu0_samps = trace.get_values("mu0", combine=True)               # shape: (S,)

print("mu_regime_samps shape:", mu_regime_samps.shape)
print("beta_samps shape:", beta_samps.shape)
print("sigma_samps shape:", sigma_samps.shape)

# =========================================
# B. 1) Raw pair differences sanity check
# =========================================
print("\n" + "="*60)
print("1) RAW PAIR DIFFERENCES CHECK")
print("="*60)
raw_summary = pair_df.groupby("regime")["d"].agg(["mean", "median", "std", "count"])
print(raw_summary)

# 可选画图
plt.figure(figsize=(8, 5))
pair_df.boxplot(column="d", by="regime")
plt.axhline(0, color="red", linestyle="--")
plt.title("Raw pair differences by regime")
plt.suptitle("")
plt.ylabel("d = treat - control")
plt.show()

# =========================================
# C. 2) Posterior mean / CI for each regime
# =========================================
print("\n" + "="*60)
print("2) POSTERIOR SUMMARY FOR EACH REGIME")
print("="*60)

posterior_mu = mu_regime_samps.mean(axis=0)
posterior_ci = np.quantile(mu_regime_samps, [0.025, 0.975], axis=0)

for i, name in enumerate(regime_order):
    print(
        f"{name}: mean={posterior_mu[i]:.4f}, "
        f"95% CrI=[{posterior_ci[0, i]:.4f}, {posterior_ci[1, i]:.4f}]"
    )

print(f"\nmu0 mean = {mu0_samps.mean():.4f}")
print(f"tau mean = {tau_samps.mean():.4f}")
print(f"sigma mean = {sigma_samps.mean():.4f}")
print(f"beta_cultivar mean = {beta_samps.mean():.4f}")

# =========================================
# D. 3) Pairwise posterior comparisons
# =========================================
print("\n" + "="*60)
print("3) PAIRWISE POSTERIOR COMPARISONS")
print("="*60)

regime_map = {name: i for i, name in enumerate(regime_order)}

def compare_regimes(name1, name2, mu_samples, regime_map):
    i = regime_map[name1]
    j = regime_map[name2]
    diff = mu_samples[:, i] - mu_samples[:, j]
    p_gt = np.mean(diff > 0)
    ci = np.quantile(diff, [0.025, 0.975])
    print(f"{name1} - {name2}:")
    print(f"  P({name1} > {name2}) = {p_gt:.3f}")
    print(f"  95% CrI = [{ci[0]:.4f}, {ci[1]:.4f}]")
    return diff

lt_minus_a = compare_regimes("LT", "A", mu_regime_samps, regime_map)
lt_minus_b = compare_regimes("LT", "B", mu_regime_samps, regime_map)
lt_minus_c = compare_regimes("LT", "C", mu_regime_samps, regime_map)
lt_minus_d = compare_regimes("LT", "D", mu_regime_samps, regime_map)

# =========================================
# E. 4) Posterior predictive check
# =========================================
print("\n" + "="*60)
print("4) POSTERIOR PREDICTIVE CHECK")
print("="*60)

S = len(beta_samps)
n = len(d)

# 抽一部分 posterior draws 做 replicate
n_ppc = min(300, S)
idx = np.random.choice(S, size=n_ppc, replace=False)

d_rep = np.zeros((n_ppc, n))

for k, s in enumerate(idx):
    mu_obs_s = mu_regime_samps[s, regime_idx] + beta_samps[s] * cultivar
    d_rep[k, :] = np.random.normal(loc=mu_obs_s, scale=sigma_samps[s], size=n)

# 比较整体 mean 和 sd
obs_mean = d.mean()
rep_mean = d_rep.mean(axis=1)

obs_sd = d.std(ddof=1)
rep_sd = d_rep.std(axis=1, ddof=1)

print(f"Observed mean(d) = {obs_mean:.4f}")
print("Posterior predictive mean(d) 95% interval =",
      np.quantile(rep_mean, [0.025, 0.975]))

print(f"Observed sd(d) = {obs_sd:.4f}")
print("Posterior predictive sd(d) 95% interval =",
      np.quantile(rep_sd, [0.025, 0.975]))

# 画图：mean(d)
plt.figure(figsize=(7, 4))
plt.hist(rep_mean, bins=30, alpha=0.7, edgecolor="black")
plt.axvline(obs_mean, color="red", linewidth=2, label="Observed mean(d)")
plt.title("Posterior predictive check: mean(d)")
plt.legend()
plt.show()

# 画图：sd(d)
plt.figure(figsize=(7, 4))
plt.hist(rep_sd, bins=30, alpha=0.7, edgecolor="black")
plt.axvline(obs_sd, color="red", linewidth=2, label="Observed sd(d)")
plt.title("Posterior predictive check: sd(d)")
plt.legend()
plt.show()

# 画图：整体分布粗比较
plt.figure(figsize=(7, 4))
plt.hist(d, bins=15, alpha=0.6, density=True, edgecolor="black", label="Observed d")
for k in range(min(20, n_ppc)):
    plt.hist(d_rep[k, :], bins=15, alpha=0.05, density=True, color="gray")
plt.title("Observed d vs posterior predictive replicates")
plt.legend()
plt.show()

# =========================================
# F. 5) Residual check
# =========================================
print("\n" + "="*60)
print("5) RESIDUAL CHECK")
print("="*60)

mu_regime_mean = mu_regime_samps.mean(axis=0)
beta_mean = beta_samps.mean()

fitted = mu_regime_mean[regime_idx] + beta_mean * cultivar
resid = d - fitted

pair_df["fitted"] = fitted
pair_df["resid"] = resid

resid_summary = pair_df.groupby("regime")["resid"].agg(["mean", "median", "std", "count"])
print(resid_summary)

# residual vs fitted
plt.figure(figsize=(7, 4))
plt.scatter(fitted, resid)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted")
plt.ylabel("Residual")
plt.title("Residual vs fitted")
plt.show()

# residual histogram
plt.figure(figsize=(7, 4))
plt.hist(resid, bins=15, edgecolor="black", alpha=0.7)
plt.axvline(0, color="red", linestyle="--")
plt.title("Residual histogram")
plt.xlabel("Residual")
plt.show()

# residual by regime
plt.figure(figsize=(8, 5))
pair_df.boxplot(column="resid", by="regime")
plt.axhline(0, color="red", linestyle="--")
plt.title("Residuals by regime")
plt.suptitle("")
plt.ylabel("Residual")
plt.show()

# =========================================
# G. 6) Shrinkage check
# =========================================
print("\n" + "="*60)
print("6) SHRINKAGE CHECK")
print("="*60)

raw_mean = pair_df.groupby("regime")["d"].mean().reindex(regime_order)
post_mean = pd.Series(posterior_mu, index=regime_order)

shrink_df = pd.DataFrame({
    "raw_mean_d": raw_mean,
    "posterior_mu": post_mean
})
print(shrink_df)

plt.figure(figsize=(7, 5))
x = np.arange(len(regime_order))
plt.scatter(x, raw_mean.values, label="raw mean(d)", s=80)
plt.scatter(x, post_mean.values, label="posterior mu_g", s=80)
for i in range(len(regime_order)):
    plt.plot([x[i], x[i]], [raw_mean.values[i], post_mean.values[i]], color="gray", alpha=0.6)
plt.xticks(x, regime_order)
plt.axhline(0, color="red", linestyle="--")
plt.ylabel("Effect")
plt.title("Shrinkage check: raw mean(d) vs posterior mu_g")
plt.legend()
plt.show()

# =========================================
# H. 7) Credible interval reasonableness
# =========================================
print("\n" + "="*60)
print("7) CREDIBLE INTERVAL CHECK")
print("="*60)

ci_width = posterior_ci[1, :] - posterior_ci[0, :]
ci_df = pd.DataFrame({
    "regime": regime_order,
    "posterior_mean": posterior_mu,
    "ci_lower": posterior_ci[0, :],
    "ci_upper": posterior_ci[1, :],
    "ci_width": ci_width
})
print(ci_df)

plt.figure(figsize=(8, 5))
for i, name in enumerate(regime_order):
    plt.plot([i, i], [posterior_ci[0, i], posterior_ci[1, i]], color="black")
    plt.scatter(i, posterior_mu[i], s=80)
plt.xticks(np.arange(len(regime_order)), regime_order)
plt.axhline(0, color="red", linestyle="--")
plt.ylabel("Posterior effect")
plt.title("Posterior means and 95% credible intervals")
plt.show()

# =========================================
# I. 8) Nonparametric sanity check
#    这里做简单的 sign-based check
# =========================================
print("\n" + "="*60)
print("8) NONPARAMETRIC SANITY CHECK")
print("="*60)

sign_check = pair_df.groupby("regime")["d"].apply(lambda x: np.mean(x > 0))
print("Proportion of positive pair differences by regime:")
print(sign_check)

# 也可以看中位数
median_check = pair_df.groupby("regime")["d"].median()
print("\nMedian pair differences by regime:")
print(median_check)

# =========================================
# J. 9) Final checklist summary
# =========================================
print("\n" + "="*60)
print("9) FINAL CHECKLIST")
print("="*60)

rep_mean_ci = np.quantile(rep_mean, [0.025, 0.975])
rep_sd_ci = np.quantile(rep_sd, [0.025, 0.975])

print("Observed mean(d) within PPC interval?:",
      rep_mean_ci[0] <= obs_mean <= rep_mean_ci[1])

print("Observed sd(d) within PPC interval?:",
      rep_sd_ci[0] <= obs_sd <= rep_sd_ci[1])

print("Residual means by regime (should be near 0):")
print(resid_summary["mean"])

print("\nShrinkage table:")
print(shrink_df)

print("\nPosterior CI table:")
print(ci_df)

print("\nNonparametric direction check:")
print(sign_check)


## C2. Healthy Fruit Weight — Bayesian Hierarchical Model (Student-t Likelihood)


In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import matplotlib.pyplot as plt

# =========
# 1. 读取 LT 数据
# =========
# Uses same data files as Parts A & B above
lt_path = "data_longterm/LTYielddata2024.csv"
lt_raw = pd.read_csv(lt_path, header=3)

# 清理列名空格
lt_raw.columns = lt_raw.columns.str.strip()

print("LT columns:")
print(list(lt_raw.columns))

lt = lt_raw.rename(columns={
    "HeatTrt": "Treatment",
    "Rotten fruit #": "rotten_count",
    "Rotten fruitWeight (g)": "rotten_weight",
    "Non-rotten fruit#": "healthy_count",
    "Non-rotten fruitWeight (g)": "healthy_weight",
    "Total #": "total_count",
    "Total Weight (g)": "total_weight"
}).copy()

lt["Cultivar"] = lt["Cultivar"].astype(str).str.strip()
lt["Treatment"] = lt["Treatment"].astype(str).str.strip()
lt["Plot"] = pd.to_numeric(lt["Plot"], errors="coerce")

# ===== outcome 改成 healthy fruit total weight =====
lt["y_healthy"] = pd.to_numeric(lt["healthy_weight"], errors="coerce")

# 构造 long-term pair_id
# 假设同 cultivar 内 control plot = OTC plot + 4
def make_lt_pair_id(row):
    plot = row["Plot"]
    if pd.isna(plot):
        return np.nan
    if row["Cultivar"] in ["St", "MQ", "Stevens"]:
        return plot if row["Treatment"] == "OTC" else plot - 4
    else:
        raise ValueError(f"Unexpected cultivar: {row['Cultivar']}")

lt["pair_index"] = lt.apply(make_lt_pair_id, axis=1)
lt["regime"] = "LT"

# 宽表：每对一行
lt_wide = (
    lt.pivot_table(
        index=["Cultivar", "pair_index", "regime"],
        columns="Treatment",
        values="y_healthy",
        aggfunc="first"
    )
    .reset_index()
)

# 只保留成对完整的
lt_wide = lt_wide.dropna(subset=["OTC", "Control"]).copy()

# pair difference
lt_wide["d"] = lt_wide["OTC"] - lt_wide["Control"]
lt_pairs = lt_wide[["Cultivar", "pair_index", "regime", "d"]].copy()

# =========
# 2. 读取 Acute 数据
# =========
acute_path = "data_acute/Acute HS-Yield_RawData 2024.xlsx"
acute = pd.read_excel(acute_path, sheet_name="Acute Heat stress", header=4)

acute.columns = acute.columns.str.strip()

print("Acute columns:")
print(list(acute.columns))

acute = acute.rename(columns={
    "Rotten fruitWeight (g)": "rotten_weight",
    "Total Weight (g)": "total_weight",
    "Non-rotten fruitWeight (g)": "healthy_weight",
    "Rotten fruit #": "rotten_count",
    "Non-rotten fruit#": "healthy_count",
    "Total #": "total_count",
    "# OTC/ plot": "rep"
}).copy()

acute = acute.dropna(subset=["Cultivar", "Treatment", "rep"])
acute["Cultivar"] = acute["Cultivar"].astype(str).str.strip()
acute["Treatment"] = acute["Treatment"].astype(str).str.strip()
acute["rep"] = pd.to_numeric(acute["rep"], errors="coerce")
acute = acute.dropna(subset=["rep"]).copy()
acute["rep"] = acute["rep"].astype(int)

# ===== outcome 改成 healthy fruit total weight =====
acute["y_healthy"] = pd.to_numeric(acute["healthy_weight"], errors="coerce")

# 只保留 A/B/C/D 和对应 control
acute = acute[acute["Treatment"].isin(["A", "B", "C", "D", "AC", "BC", "CC", "DC"])].copy()

# treatment family
acute["regime"] = acute["Treatment"].replace({
    "AC": "A",
    "BC": "B",
    "CC": "C",
    "DC": "D"
})

acute["group"] = np.where(
    acute["Treatment"].isin(["A", "B", "C", "D"]),
    "treat",
    "control"
)

acute_wide = (
    acute.pivot_table(
        index=["Cultivar", "rep", "regime"],
        columns="group",
        values="y_healthy",
        aggfunc="first"
    )
    .reset_index()
)

# 只保留成对完整的
acute_wide = acute_wide.dropna(subset=["treat", "control"]).copy()

acute_wide["d"] = acute_wide["treat"] - acute_wide["control"]
acute_pairs = acute_wide[["Cultivar", "rep", "regime", "d"]].copy()
acute_pairs = acute_pairs.rename(columns={"rep": "pair_index"})

# =========
# 3. 合并 pair-level 数据
# =========
pair_df = pd.concat([lt_pairs, acute_pairs], ignore_index=True)

# 统一 cultivar 名称
pair_df["Cultivar"] = pair_df["Cultivar"].replace({
    "Stevens": "St"
})

pair_df = pair_df.dropna(subset=["Cultivar", "d"]).copy()

# 编码
regime_order = ["LT", "A", "B", "C", "D"]
pair_df["regime"] = pd.Categorical(pair_df["regime"], categories=regime_order, ordered=True)
pair_df = pair_df.dropna(subset=["regime"]).copy()
pair_df["regime_idx"] = pair_df["regime"].cat.codes

pair_df["cultivar_code"] = pair_df["Cultivar"].map({
    "St": 0,
    "MQ": 1
})

pair_df = pair_df.dropna(subset=["cultivar_code"]).copy()
pair_df["cultivar_code"] = pair_df["cultivar_code"].astype(int)

print("\nPair-level data:")
print(pair_df)

# =========
# 4. 贝叶斯层级模型（修正版：non-centered + Student-t + regime-specific sigma）
# =========
d = pair_df["d"].values.astype(float)
regime_idx = pair_df["regime_idx"].values
cultivar = pair_df["cultivar_code"].values
n_regimes = len(regime_order)

with pm.Model() as model:
    # -------------------------
    # Hyperpriors
    # -------------------------
    mu0 = pm.Normal("mu0", mu=0, sigma=80)
    tau = pm.HalfNormal("tau", sigma=80)

    # non-centered regime effects
    z_regime = pm.Normal("z_regime", mu=0, sigma=1, shape=n_regimes)
    mu_regime = pm.Deterministic("mu_regime", mu0 + z_regime * tau)

    # cultivar main effect
    beta_cultivar = pm.Normal("beta_cultivar", mu=0, sigma=80)

    # regime-specific residual scales
    sigma_regime = pm.HalfNormal("sigma_regime", sigma=150, shape=n_regimes)

    # Student-t degrees of freedom
    nu_minus_two = pm.Exponential("nu_minus_two", lam=1/10)
    nu = pm.Deterministic("nu", nu_minus_two + 2)

    # mean model
    mu_obs = mu_regime[regime_idx] + beta_cultivar * cultivar

    # likelihood
    d_obs = pm.StudentT(
        "d_obs",
        nu=nu,
        mu=mu_obs,
        sigma=sigma_regime[regime_idx],
        observed=d
    )

    trace = pm.sample(
        draws=3000,
        tune=4000,
        chains=4,
        cores=1,
        init="jitter+adapt_diag",
        target_accept=0.99,
        random_seed=123,
        return_inferencedata=False,
        compute_convergence_checks=False
    )

# =========
# 5. 查看结果
# =========
mu_samples = trace.get_values("mu_regime", combine=True)               # shape: (S, 5)
beta_samples = trace.get_values("beta_cultivar", combine=True)        # shape: (S,)
sigma_regime_samples = trace.get_values("sigma_regime", combine=True) # shape: (S, 5)
tau_samples = trace.get_values("tau", combine=True)
mu0_samples = trace.get_values("mu0", combine=True)
nu_samples = trace.get_values("nu", combine=True)

posterior_mu = mu_samples.mean(axis=0)
for name, val in zip(regime_order, posterior_mu):
    print(f"{name}: {val:.4f}")

print("beta_cultivar mean =", beta_samples.mean())
print("sigma_regime mean =", sigma_regime_samples.mean(axis=0))
print("tau mean =", tau_samples.mean())
print("mu0 mean =", mu0_samples.mean())
print("nu mean =", nu_samples.mean())

print("mu_regime_samps shape:", mu_samples.shape)
print("beta_samps shape:", beta_samples.shape)
print("sigma_regime_samps shape:", sigma_regime_samples.shape)
print("nu_samps shape:", nu_samples.shape)

# =========================================
# B. 1) Raw pair differences sanity check
# =========================================
print("\n" + "="*60)
print("1) RAW PAIR DIFFERENCES CHECK (HEALTHY FRUIT WEIGHT)")
print("="*60)
raw_summary = pair_df.groupby("regime")["d"].agg(["mean", "median", "std", "count"])
print(raw_summary)

plt.figure(figsize=(8, 5))
pair_df.boxplot(column="d", by="regime")
plt.axhline(0, color="red", linestyle="--")
plt.title("Raw pair differences by regime (healthy fruit total weight)")
plt.suptitle("")
plt.ylabel("d = treat - control (healthy fruit total weight)")
plt.show()

# =========================================
# C. 2) Posterior mean / CI for each regime
# =========================================
print("\n" + "="*60)
print("2) POSTERIOR SUMMARY FOR EACH REGIME")
print("="*60)

posterior_ci = np.quantile(mu_samples, [0.025, 0.975], axis=0)

for i, name in enumerate(regime_order):
    print(
        f"{name}: mean={posterior_mu[i]:.4f}, "
        f"95% CrI=[{posterior_ci[0, i]:.4f}, {posterior_ci[1, i]:.4f}]"
    )

print(f"\nmu0 mean = {mu0_samples.mean():.4f}")
print(f"tau mean = {tau_samples.mean():.4f}")
print(f"nu mean = {nu_samples.mean():.4f}")
print(f"beta_cultivar mean = {beta_samples.mean():.4f}")

sigma_regime_mean = sigma_regime_samples.mean(axis=0)
print("sigma_regime means by regime:")
for i, name in enumerate(regime_order):
    print(f"  {name}: {sigma_regime_mean[i]:.4f}")

# =========================================
# D. 3) Pairwise posterior comparisons
# =========================================
print("\n" + "="*60)
print("3) PAIRWISE POSTERIOR COMPARISONS")
print("="*60)

regime_map = {name: i for i, name in enumerate(regime_order)}

def compare_regimes(name1, name2, mu_samples, regime_map):
    i = regime_map[name1]
    j = regime_map[name2]
    diff = mu_samples[:, i] - mu_samples[:, j]
    p_gt = np.mean(diff > 0)
    ci = np.quantile(diff, [0.025, 0.975])
    print(f"{name1} - {name2}:")
    print(f"  P({name1} > {name2}) = {p_gt:.3f}")
    print(f"  95% CrI = [{ci[0]:.4f}, {ci[1]:.4f}]")
    return diff

lt_minus_a = compare_regimes("LT", "A", mu_samples, regime_map)
lt_minus_b = compare_regimes("LT", "B", mu_samples, regime_map)
lt_minus_c = compare_regimes("LT", "C", mu_samples, regime_map)
lt_minus_d = compare_regimes("LT", "D", mu_samples, regime_map)

# =========================================
# E. 4) Posterior predictive check
# =========================================
print("\n" + "="*60)
print("4) POSTERIOR PREDICTIVE CHECK")
print("="*60)

S = len(beta_samples)
n = len(d)

n_ppc = min(300, S)
idx = np.random.choice(S, size=n_ppc, replace=False)

d_rep = np.zeros((n_ppc, n))

for k, s in enumerate(idx):
    mu_obs_s = mu_samples[s, regime_idx] + beta_samples[s] * cultivar
    sigma_obs_s = sigma_regime_samples[s, regime_idx]
    nu_s = nu_samples[s]

    d_rep[k, :] = mu_obs_s + sigma_obs_s * np.random.standard_t(df=nu_s, size=n)

obs_mean = d.mean()
rep_mean = d_rep.mean(axis=1)

obs_sd = d.std(ddof=1)
rep_sd = d_rep.std(axis=1, ddof=1)

print(f"Observed mean(d) = {obs_mean:.4f}")
print("Posterior predictive mean(d) 95% interval =",
      np.quantile(rep_mean, [0.025, 0.975]))

print(f"Observed sd(d) = {obs_sd:.4f}")
print("Posterior predictive sd(d) 95% interval =",
      np.quantile(rep_sd, [0.025, 0.975]))

plt.figure(figsize=(7, 4))
plt.hist(rep_mean, bins=30, alpha=0.7, edgecolor="black")
plt.axvline(obs_mean, color="red", linewidth=2, label="Observed mean(d)")
plt.title("Posterior predictive check: mean(d)")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(rep_sd, bins=30, alpha=0.7, edgecolor="black")
plt.axvline(obs_sd, color="red", linewidth=2, label="Observed sd(d)")
plt.title("Posterior predictive check: sd(d)")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(d, bins=15, alpha=0.6, density=True, edgecolor="black", label="Observed d")
for k in range(min(20, n_ppc)):
    plt.hist(d_rep[k, :], bins=15, alpha=0.05, density=True, color="gray")
plt.title("Observed d vs posterior predictive replicates")
plt.legend()
plt.show()

# =========================================
# F. 5) Residual check
# =========================================
print("\n" + "="*60)
print("5) RESIDUAL CHECK")
print("="*60)

mu_regime_mean = mu_samples.mean(axis=0)
beta_mean = beta_samples.mean()

fitted = mu_regime_mean[regime_idx] + beta_mean * cultivar
resid = d - fitted

pair_df["fitted"] = fitted
pair_df["resid"] = resid

resid_summary = pair_df.groupby("regime")["resid"].agg(["mean", "median", "std", "count"])
print(resid_summary)

plt.figure(figsize=(7, 4))
plt.scatter(fitted, resid)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted")
plt.ylabel("Residual")
plt.title("Residual vs fitted")
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(resid, bins=15, edgecolor="black", alpha=0.7)
plt.axvline(0, color="red", linestyle="--")
plt.title("Residual histogram")
plt.xlabel("Residual")
plt.show()

plt.figure(figsize=(8, 5))
pair_df.boxplot(column="resid", by="regime")
plt.axhline(0, color="red", linestyle="--")
plt.title("Residuals by regime")
plt.suptitle("")
plt.ylabel("Residual")
plt.show()

# =========================================
# G. 6) Shrinkage check
# =========================================
print("\n" + "="*60)
print("6) SHRINKAGE CHECK")
print("="*60)

raw_mean = pair_df.groupby("regime")["d"].mean().reindex(regime_order)
post_mean = pd.Series(posterior_mu, index=regime_order)

shrink_df = pd.DataFrame({
    "raw_mean_d": raw_mean,
    "posterior_mu": post_mean
})
print(shrink_df)

plt.figure(figsize=(7, 5))
x = np.arange(len(regime_order))
plt.scatter(x, raw_mean.values, label="raw mean(d)", s=80)
plt.scatter(x, post_mean.values, label="posterior mu_g", s=80)
for i in range(len(regime_order)):
    plt.plot([x[i], x[i]], [raw_mean.values[i], post_mean.values[i]], color="gray", alpha=0.6)
plt.xticks(x, regime_order)
plt.axhline(0, color="red", linestyle="--")
plt.ylabel("Effect")
plt.title("Shrinkage check: raw mean(d) vs posterior mu_g")
plt.legend()
plt.show()

# =========================================
# H. 7) Credible interval reasonableness
# =========================================
print("\n" + "="*60)
print("7) CREDIBLE INTERVAL CHECK")
print("="*60)

ci_width = posterior_ci[1, :] - posterior_ci[0, :]
ci_df = pd.DataFrame({
    "regime": regime_order,
    "posterior_mean": posterior_mu,
    "ci_lower": posterior_ci[0, :],
    "ci_upper": posterior_ci[1, :],
    "ci_width": ci_width
})
print(ci_df)

plt.figure(figsize=(8, 5))
for i, name in enumerate(regime_order):
    plt.plot([i, i], [posterior_ci[0, i], posterior_ci[1, i]], color="black")
    plt.scatter(i, posterior_mu[i], s=80)
plt.xticks(np.arange(len(regime_order)), regime_order)
plt.axhline(0, color="red", linestyle="--")
plt.ylabel("Posterior effect")
plt.title("Posterior means and 95% credible intervals")
plt.show()

# =========================================
# I. 8) Nonparametric sanity check
# =========================================
print("\n" + "="*60)
print("8) NONPARAMETRIC SANITY CHECK")
print("="*60)

sign_check = pair_df.groupby("regime")["d"].apply(lambda x: np.mean(x > 0))
print("Proportion of positive pair differences by regime:")
print(sign_check)

median_check = pair_df.groupby("regime")["d"].median()
print("\nMedian pair differences by regime:")
print(median_check)

# =========================================
# J. 9) Final checklist summary
# =========================================
print("\n" + "="*60)
print("9) FINAL CHECKLIST")
print("="*60)

rep_mean_ci = np.quantile(rep_mean, [0.025, 0.975])
rep_sd_ci = np.quantile(rep_sd, [0.025, 0.975])

print("Observed mean(d) within PPC interval?:",
      rep_mean_ci[0] <= obs_mean <= rep_mean_ci[1])

print("Observed sd(d) within PPC interval?:",
      rep_sd_ci[0] <= obs_sd <= rep_sd_ci[1])

print("Residual means by regime (should be near 0):")
print(resid_summary["mean"])

print("\nShrinkage table:")
print(shrink_df)

print("\nPosterior CI table:")
print(ci_df)

print("\nNonparametric direction check:")
print(sign_check)


---

## Summary of All Outputs



**Part A (Long-term):**

- Healthy weight: Δ = −61.7g, p = 0.340, not significant

- Rotten proportion: Δ = −4.0%, p = 0.015, significant (OTC reduced rot)



**Part B (Acute):**

- Healthy weight: Δ = −155.3g, p = 0.001, significant

- Rotten proportion: Δ = +10.8%, p < 0.0001, significant

- Dose-response slope: 12.1 g/pulse, p = 0.751, not significant



**Part C (Cross-experiment):**

- Healthy weight: Treatment C has entire 95% CrI below zero; P(LT > C) = 0.972

- Rotten proportion: Treatment D has entire 95% CrI above zero; P(LT > D) = 0.007

- Acute extreme heat causes greater damage than gradual long-term warming
